In [ ]:
!pip install Google-Images-Search > /dev/null 2>&1

In [ ]:
from google.colab import drive
import pandas as pd
import re
import os
import os
import re
import torch
from PIL import Image
from transformers import AutoProcessor, CLIPModel
import torch.nn as nn
import os
import re
import torch
import pandas as pd
from PIL import Image
import torch.nn as nn
from transformers import AutoProcessor, CLIPModel, AutoImageProcessor, AutoModel
import os
import json
import pandas as pd
from google.colab import drive
from google_images_search import GoogleImagesSearch
import time
import requests

In [ ]:
# mount drive
drive.mount('/content/drive')

In [ ]:
# get already generated google images
df_google_generated_images = pd.read_csv('/content/drive/MyDrive/Thesis/generated_google_images.csv')

In [ ]:
# strip the paths from bbc_images_0006_139_2.jpg to ./bbc/images/0006/139.jpg
def strip_path(path):
    filename = path.split('/')[-1]
    parts = filename.replace('.jpg', '').split('_')
    prefix = '_'.join(parts[:-4])
    return f"./{prefix}/images/{parts[-3]}/{parts[-2]}.jpg"


df_generated_images = pd.read_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')
df_generated_images['image_path_stripped'] = df_generated_images['image_path'].apply(strip_path)
grouped_counts = df_generated_images.groupby('image_path_stripped').size().reset_index(name='count')
fullgeneratedimages = grouped_counts[grouped_counts['count'] >= 5]
fullgeneratedimages


In [ ]:
# get the remaining generated images
df_remaining = fullgeneratedimages[
    ~fullgeneratedimages['image_path_stripped'].isin(df_google_generated_images['image_path_stripped'])
]
images_paths = df_remaining['image_path_stripped']

## download top 5 google images

In [ ]:
# access metadata of original images (captions etc.)
data_path = '/content/drive/MyDrive/Thesis/images_dataset_data_description/data.json'
data = json.load(open(data_path))
df = pd.DataFrame(data)

# google image search keys
API_KEY = 'XXXXXX'
CX = 'XXXXXXX'
gis = GoogleImagesSearch(API_KEY, CX)

# get top 5 Google Images using captions
def get_top5_google_images(query):
    search_params = {
        'q': query,
        'num': 10,
        'safe': 'off',
        'fileType': 'jpg|png',
        'imgType': 'photo',
    }

    gis.search(search_params=search_params)

    results = []
    for image in gis.results():
        results.append({
            'image_url': image.url
        })
    return results

def fetch_and_save_google_images(image_path):
    print(f"Processing: {image_path}")
    # clean path
    image_path_strip = image_path.strip().lstrip('./')

    # get caption
    try:
        caption_text = df[df['image_path'] == image_path]['caption'].values[0]
        print(f"Caption: {caption_text}")
    except IndexError:
        print(f"No caption found for {image_path}")
        return

    # fetch images
    try:
        images = get_top5_google_images(caption_text)
        print(f"Found {len(images)} images")
    except Exception as e:
        print(f"Failed to fetch images for: {caption_text}\nError: {e}")
        return

    # Save images
    save_base_folder = '/content/drive/MyDrive/Thesis/images_google'
    os.makedirs(save_base_folder, exist_ok=True)

    parts = image_path_strip.split('/')
    filename_prefix = f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}"

    for idx, img in enumerate(images, 1):
        image_url = img['image_url']
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            save_path = f"{save_base_folder}/{filename_prefix}_{idx}_google.jpg"
            with open(save_path, 'wb') as f:
                f.write(response.content)
            print(f"Saved: {save_path}")
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")



In [ ]:
# Run the function
count_num = 0
for image in images_paths:
  print(count_num)
  print(image)
  fetch_and_save_google_images(image)
  count_num+=1